# Neural Networks and Deep Learning — Seminar 9: Graph neural networks
Prof. dr. Daniel Traian Pele — Bucharest University of Economic Studies (ASE); Romanian Academy, Institute for Economic Forecasting (IPE)

[Open in Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L9_seminar.ipynb)

Seminar tasks on real data from `../data/clean` (see `../data/SOURCES.md`). Write your code in the empty cells under each task; keep the seed fixed so that your numbers can be compared.

In [ ]:
# Google Colab: copy the course repository, so that the data and the saved results are found
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Setup: the Elliptic graph

In [ ]:
import subprocess, sys, urllib.request
try:
    import torch_geometric
except ImportError:                                   # Google Colab: install PyTorch Geometric
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
import torch, torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from torch_geometric.utils import to_undirected
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score
RAW = '../data/raw/elliptic/'
os.makedirs(RAW, exist_ok=True)
for fn in ['elliptic_txs_features.csv.zip', 'elliptic_txs_edgelist.csv.zip', 'elliptic_txs_classes.csv.zip']:
    if not os.path.exists(RAW + fn):                  # about 146 MB, served by PyTorch Geometric
        urllib.request.urlretrieve('https://data.pyg.org/datasets/elliptic/' + fn, RAW + fn)
feat = pd.read_csv(RAW + 'elliptic_txs_features.csv.zip', header=None)
lab = pd.read_csv(RAW + 'elliptic_txs_classes.csv.zip')
edg = pd.read_csv(RAW + 'elliptic_txs_edgelist.csv.zip')
txid, step = feat[0].values, feat[1].values.astype(int)
X = feat.iloc[:, 2:].values.astype(np.float32)               # 93 local + 72 aggregated features
cls_map = dict(zip(lab.txId, lab['class'].astype(str)))
y = np.array([{'1': 1, '2': 0}.get(cls_map[t], -1) for t in txid])   # 1 illicit, 0 licit, -1 unknown
pos = {t: i for i, t in enumerate(txid)}
E = np.array([[pos[a] for a in edg.txId1], [pos[b] for b in edg.txId2]])
lab_m = y >= 0
train_m, test_m = lab_m & (step <= 34), lab_m & (step > 34)
fit_m, val_m = lab_m & (step < 30), lab_m & (step >= 30) & (step <= 34)
print('nodes', len(y), 'edges', E.shape[1], 'illicit', (y == 1).sum(), 'licit', (y == 0).sum(), 'unknown', (y == -1).sum())
print('labelled training', train_m.sum(), 'labelled test', test_m.sum(), 'illicit in test', (y[test_m] == 1).sum())

In [ ]:
mu, sd = X[step <= 34].mean(0), X[step <= 34].std(0) + 1e-6      # training steps only: no leakage
Xs = np.clip((X - mu) / sd, -10, 10).astype(np.float32)
ei = to_undirected(torch.tensor(E, dtype=torch.long))
xt, yt = torch.tensor(Xs), torch.tensor(np.where(y < 0, 0, y))
W = torch.tensor([0.3, 0.7])                                       # class weights of Weber et al. (2019)

def metrics(p, mask=test_m, thr=0.5):
    yy, pp = y[mask], p[mask]
    return dict(f1=round(f1_score(yy, pp >= thr), 3), prec=round(precision_score(yy, pp >= thr), 3),
                rec=round(recall_score(yy, pp >= thr), 3), prauc=round(average_precision_score(yy, pp), 3))

class GNN(torch.nn.Module):
    def __init__(self, kind, d_in, h=100, layers=2, heads=4, drop=0.2):
        super().__init__()
        self.kind, self.drop = kind, drop
        mk = {'gcn': lambda a: GCNConv(a, h), 'sage': lambda a: SAGEConv(a, h), 'gat': lambda a: GATConv(a, h // heads, heads=heads),
              'mlp': lambda a: torch.nn.Linear(a, h)}[kind]
        self.convs = torch.nn.ModuleList([mk(d_in if i == 0 else h) for i in range(layers)])
        self.out = torch.nn.Linear(h, 2)
    def embed(self, x, ei):
        for c in self.convs:
            x = F.dropout(F.relu(c(x) if self.kind == 'mlp' else c(x, ei)), self.drop, self.training)
        return x
    def forward(self, x, ei):
        return self.out(self.embed(x, ei))

def train(kind, layers=2, epochs=200, patience=30, seed=1):
    torch.manual_seed(seed)
    m = GNN(kind, xt.shape[1], layers=layers); opt = torch.optim.Adam(m.parameters(), lr=0.003, weight_decay=5e-4)
    best, state, wait = -1, None, 0
    fm = torch.tensor(fit_m)
    for ep in range(epochs):
        m.train(); opt.zero_grad()
        loss = F.cross_entropy(m(xt, ei)[fm], yt[fm], weight=W); loss.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            p = torch.softmax(m(xt, ei), 1)[:, 1].numpy()
        ap = average_precision_score(y[val_m], p[val_m])
        if ap > best:
            best, wait, state = ap, 0, {k: v.clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    m.load_state_dict(state); m.eval()
    with torch.no_grad():
        return torch.softmax(m(xt, ei), 1)[:, 1].numpy(), m.embed(xt, ei).numpy()

**Task 1.** Train a logistic regression on the 93 local features only, then on all 165 features. How much do the 72 neighbourhood aggregates add to the illicit-class F1 and the PR-AUC?

In [ ]:
# your code here

**Task 2.** Train the GCN and an MLP with the same layers (no edges). Which one has the larger PR-AUC on the test steps? What does the difference measure?

In [ ]:
# your code here

**Task 3.** Random forest + embeddings: add the 100 hidden GCN outputs to the 165 features and train a random forest (100 trees, 50 features per split). Compare its F1 with the plain forest.

In [ ]:
# your code here

**Task 4.** Threshold choice: for the GCN scores, find the threshold that maximises the illicit F1 on the validation steps 30–34, then apply it to the test steps. Does F1 improve over 0.5?

In [ ]:
# your code here

**Task 5.** Drift: compute the illicit F1 of the random forest separately for test steps 35–42 and 43–49. What happened at step 43, and what would you change in production?

In [ ]:
# your code here